In [29]:
!pip install torch transformers datasets pytesseract opencv-python rouge-score pandas numpy==1.26.4

In [30]:
import re
import os
import cv2
import pytesseract
import pandas as pd

from datasets import load_dataset
from transformers import pipeline
from rouge_score import rouge_scorer

TEXT CLEANING

In [31]:
def clean_text(text):
    if not text:
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

OCR 

In [32]:
def extract_text_with_debug(image_path):
    if not os.path.exists(image_path):
        raise FileNotFoundError("Image not found")

    img = cv2.imread(image_path)

    raw_text = pytesseract.image_to_string(img)

    img = cv2.resize(img, None, fx=2, fy=2)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.medianBlur(gray, 3)

    thresh = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        11, 2
    )

    processed_text = pytesseract.image_to_string(thresh)

    return raw_text, processed_text

In [33]:
def lstm_summary(text):
    sentences = text.split(".")
    return ". ".join(sentences[:2])

In [34]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")

Device set to use cpu
Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


In [35]:
def generate_summary(text):
    if not text:
        return ""

    input_length = len(text.split())

    # Dynamic max_length (VERY IMPORTANT)
    max_len = max(20, int(input_length * 0.5))
    min_len = max(10, int(input_length * 0.2))

    try:
        summary = summarizer(
            text,
            max_length=max_len,
            min_length=min_len,
            do_sample=False
        )
        return summary[0]['summary_text']
    except:
        return text

In [36]:
def extract_entities(text):
    results = ner(text)

    return [
        {
            "text": r["word"],
            "label": r["entity_group"],
            "confidence": round(r["score"], 3)
        }
        for r in results
    ]


def map_medical_entities(entities):
    mapped = {"DISEASE": [], "DRUG": [], "SYMPTOM": []}

    for e in entities:
        word = e["text"].lower()

        if "diabetes" in word:
            mapped["DISEASE"].append(word)
        elif "insulin" in word:
            mapped["DRUG"].append(word)
        elif "sugar" in word or "fever" in word:
            mapped["SYMPTOM"].append(word)

    return mapped

In [48]:
def run_pipeline_text(text):
    cleaned = clean_text(text)
    summary = generate_summary(cleaned)
    entities = extract_entities(summary)
    mapped = map_medical_entities(entities)

    print("\n" + "="*60)
    print("📝 RAW REPORT:\n")
    print(text)

    print("\n" + "="*60)
    print("🧹 CLEANED TEXT:\n")
    print(cleaned)

    print("\n" + "="*60)
    print("📄 SUMMARY:\n")
    print(summary)

    print("\n" + "="*60)
    print("🔍 ENTITIES:\n")
    for e in entities:
        print(e)

    print("\n" + "="*60)
    print("🏥 MEDICAL ENTITIES:\n")
    for k, v in mapped.items():
        print(f"{k}: {v}")

    print("\n" + "="*60)

    return summary

In [ ]:
def run_pipeline_image(image_path):
    raw, processed = extract_text_with_debug(image_path)

    print("\n🖼️ BEFORE OCR:\n")
    print(raw)

    print("\n🧹 AFTER OCR:\n")
    print(processed)

    return run_pipeline_text(processed)

In [39]:
dataset = load_dataset("cnn_dailymail", "3.0.0")
small_data = dataset["test"].select(range(20))

BART EVALUATION

In [40]:
references = []
predictions = []

for sample in small_data:
    text = sample["article"][:1000]
    ref = sample["highlights"]

    pred = generate_summary(text)

    references.append(ref)
    predictions.append(pred)

ROUGE EVALUATION

In [41]:
scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)

scores = [scorer.score(r, p) for r, p in zip(references, predictions)]

print("BART evaluated on dataset")

BART evaluated on dataset


LSTM vs BART COMPARISON

In [42]:
def compare_models(text, reference):
    lstm_out = lstm_summary(text)
    bart_out = generate_summary(text)

    s1 = scorer.score(reference, lstm_out)
    s2 = scorer.score(reference, bart_out)

    print("\n COMPARISON\n")

    print("LSTM ROUGE-L:", round(s1['rougeL'].fmeasure, 3))
    print("BART ROUGE-L:", round(s2['rougeL'].fmeasure, 3))

In [43]:
text = """
Patient is a 50 year old male suffering from diabetes for the past 5 years.
He has high blood sugar levels and frequent fatigue.
Doctor prescribed insulin and lifestyle changes.
Regular monitoring of glucose levels is advised.
"""

In [44]:
run_pipeline_text(text)

,Raw Report,Summary,Entities
0,\nPatient is a 50 year old male suffering from...,patient is a 50 year old male suffering from ...,"{'DISEASE': [], 'DRUG': [], 'SYMPTOM': []}"


In [45]:
# run_pipeline_image("sample_prescription.jpg")

In [46]:
compare_models(
    "Patient has diabetes. Insulin given.",
    "Patient treated for diabetes using insulin."
)

Your max_length is set to 20, but your input_length is only 10. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=5)



 COMPARISON

LSTM ROUGE-L: 0.545
BART ROUGE-L: 0.545


In [47]:
print("LSTM:", lstm_summary(text))
print("BART:", generate_summary(text))

LSTM: 
Patient is a 50 year old male suffering from diabetes for the past 5 years. 
He has high blood sugar levels and frequent fatigue
BART: Patient is a 50 year old male suffering from diabetes for the past 5 years.
